# Alibaba O2O Coupon Data Audit

This notebook audits the raw offline and online Alibaba O2O coupon datasets before any cleaning, transformation, feature engineering, or modeling.

Goals:
- Understand the structure and meaning of each dataset
- Inspect missing values and data types
- Identify users, merchants, coupons, and event types
- Understand the available time range
- Measure overlap between online and offline users
- Investigate duplicate-looking records and unusual values

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_DIR = Path("../data/raw")

offline_path = RAW_DIR / "ccf_offline_stage1_train.csv"
online_path = RAW_DIR / "ccf_online_stage1_train.csv"

offline = pd.read_csv(offline_path)
online = pd.read_csv(online_path)

## 1. Basic Dataset Structure

Check the number of rows and columns and preview the raw records.

In [2]:
print("Offline shape:", offline.shape)
print("Online shape:", online.shape)

Offline shape: (1754884, 7)
Online shape: (11429826, 7)


In [3]:
display(offline.head())

,User_id,Merchant_id,Coupon_id,Discount_rate,Distance,Date_received,Date
0,1439408,2632,NaN,NaN,0.0,NaN,20160217.0
1,1439408,4663,11002.0,150:20,1.0,20160528.0,NaN
2,1439408,2632,8591.0,20:1,0.0,20160217.0,NaN
3,1439408,2632,1078.0,20:1,0.0,20160319.0,NaN
4,1439408,2632,8591.0,20:1,0.0,20160613.0,NaN


In [4]:
display(online.head())

,User_id,Merchant_id,Action,Coupon_id,Discount_rate,Date_received,Date
0,13740231,18907,2,100017492,500:50,20160513.0,NaN
1,13740231,34805,1,NaN,NaN,NaN,20160321.0
2,14336199,18907,0,NaN,NaN,NaN,20160618.0
3,14336199,18907,0,NaN,NaN,NaN,20160618.0
4,14336199,18907,0,NaN,NaN,NaN,20160618.0


## 2. Columns and Data Types

Inspect how pandas interpreted each raw column before making any conversions.

In [5]:
print("OFFLINE")
print(offline.dtypes)

print("\nONLINE")
print(online.dtypes)

OFFLINE
User_id            int64
Merchant_id        int64
Coupon_id        float64
Discount_rate        str
Distance         float64
Date_received    float64
Date             float64
dtype: object

ONLINE
User_id            int64
Merchant_id        int64
Action             int64
Coupon_id            str
Discount_rate        str
Date_received    float64
Date             float64
dtype: object


## 3. Missing Values

Measure missingness in each field. Missing values may have business meaning, such as a coupon that was received but never redeemed.

In [6]:
offline_missing = offline.isna().sum().to_frame("missing_count")
offline_missing["missing_pct"] = (
    offline_missing["missing_count"] / len(offline) * 100
)

display(offline_missing)

,missing_count,missing_pct
User_id,0,0.000000
Merchant_id,0,0.000000
Coupon_id,701602,39.979964
Discount_rate,701602,39.979964
Distance,106003,6.040456
Date_received,701602,39.979964
Date,977900,55.724481


In [7]:
online_missing = online.isna().sum().to_frame("missing_count")
online_missing["missing_pct"] = (
    online_missing["missing_count"] / len(online) * 100
)

display(online_missing)

,missing_count,missing_pct
User_id,0,0.000000
Merchant_id,0,0.000000
Action,0,0.000000
Coupon_id,10557469,92.367714
Discount_rate,10557469,92.367714
Date_received,10557469,92.367714
Date,655898,5.738478


## 4. Unique Users, Merchants, and Coupons

Measure the number of distinct entities in each channel.

In [8]:
print("OFFLINE")
print("Users:", offline["User_id"].nunique())
print("Merchants:", offline["Merchant_id"].nunique())
print("Coupons:", offline["Coupon_id"].nunique())

print("\nONLINE")
print("Users:", online["User_id"].nunique())
print("Merchants:", online["Merchant_id"].nunique())
print("Coupons:", online["Coupon_id"].nunique())

OFFLINE
Users: 539438
Merchants: 8415
Coupons: 9738

ONLINE
Users: 762858
Merchants: 7999
Coupons: 27747


## 5. Online Action Distribution

The online dataset contains explicit user actions. Inspect how many records correspond to each action type.

In [9]:
online["Action"].value_counts(dropna=False).sort_index()

Action
0    9401780
1    1372148
2     655898
Name: count, dtype: int64

## 6. Discount Rate Formats

Inspect the raw discount values before deciding how they should be parsed.

In [10]:
offline["Discount_rate"].value_counts(dropna=False).head(30)

Discount_rate
NaN       701602
30:5      270712
100:10    182554
200:20    111046
20:5       91013
20:1       51705
50:5       47379
100:30     38196
200:30     29327
300:30     28979
50:10      28452
10:5       25925
0.95       20568
10:1       17842
30:1       17654
150:20     17437
100:20     14297
30:10      12692
50:20       8203
0.9         8085
200:50      5585
150:10      5325
100:5       5053
0.8         3441
50:1        3354
5:1         2526
100:50      1774
150:30       654
0.85         649
200:10       575
Name: count, dtype: int64

In [11]:
online["Discount_rate"].value_counts(dropna=False).head(30)

Discount_rate
NaN        10557469
fixed        131546
50:5          70964
20:10         50631
30:20         45889
5:1           42828
100:10        42047
10:5          40954
50:1          34792
200:20        34568
150:10        33615
100:50        32607
500:50        29680
50:10         28312
300:50        21655
30:1          17022
200:10        15976
300:30        15658
100:5         15141
150:50        14681
150:20        13016
50:30         12977
300:20        11357
150:5         10296
300:10         7247
10:1           6997
800:50         6732
800:100        5799
20:1           5739
200:30         5398
Name: count, dtype: int64

## 7. Distance Distribution

Inspect the offline distance field and its missing values.

In [12]:
offline["Distance"].value_counts(dropna=False).sort_index()

Distance
0.0     826070
1.0     227221
2.0     118413
3.0      76598
4.0      55085
5.0      41452
6.0      32483
7.0      25681
8.0      21436
9.0      17958
10.0    206484
NaN     106003
Name: count, dtype: int64

## 8. Date Ranges

Convert copies of the date fields to datetime so the available historical period can be measured without modifying the original columns.

In [13]:
offline["Date_received_parsed"] = pd.to_datetime(
    offline["Date_received"],
    format="%Y%m%d",
    errors="coerce"
)

offline["Date_parsed"] = pd.to_datetime(
    offline["Date"],
    format="%Y%m%d",
    errors="coerce"
)

online["Date_received_parsed"] = pd.to_datetime(
    online["Date_received"],
    format="%Y%m%d",
    errors="coerce"
)

online["Date_parsed"] = pd.to_datetime(
    online["Date"],
    format="%Y%m%d",
    errors="coerce"
)

In [14]:
print("OFFLINE")
print(
    "Coupon receipt range:",
    offline["Date_received_parsed"].min(),
    "to",
    offline["Date_received_parsed"].max()
)
print(
    "Purchase/activity range:",
    offline["Date_parsed"].min(),
    "to",
    offline["Date_parsed"].max()
)

print("\nONLINE")
print(
    "Coupon receipt range:",
    online["Date_received_parsed"].min(),
    "to",
    online["Date_received_parsed"].max()
)
print(
    "Activity range:",
    online["Date_parsed"].min(),
    "to",
    online["Date_parsed"].max()
)

OFFLINE
Coupon receipt range: 2016-01-01 00:00:00 to 2016-06-15 00:00:00
Purchase/activity range: 2016-01-01 00:00:00 to 2016-06-30 00:00:00

ONLINE
Coupon receipt range: 2016-01-01 00:00:00 to 2016-06-15 00:00:00
Activity range: 2016-01-01 00:00:00 to 2016-06-30 00:00:00


## 9. Offline Event Types

The offline file contains multiple kinds of behavior in the same table. Classify rows based on whether a coupon was received and whether a purchase occurred.

In [15]:
offline["event_type"] = "other"

offline.loc[
    offline["Coupon_id"].notna()
    & offline["Date_received"].notna()
    & offline["Date"].notna(),
    "event_type"
] = "coupon_redeemed"

offline.loc[
    offline["Coupon_id"].notna()
    & offline["Date_received"].notna()
    & offline["Date"].isna(),
    "event_type"
] = "coupon_received_not_redeemed"

offline.loc[
    offline["Coupon_id"].isna()
    & offline["Date"].notna(),
    "event_type"
] = "normal_purchase"

offline["event_type"].value_counts(dropna=False)

event_type
coupon_received_not_redeemed    977900
normal_purchase                 701602
coupon_redeemed                  75382
Name: count, dtype: int64

## 10. Online and Offline User Overlap

Users can appear in both channels even though merchant identifiers should be treated separately.

In [16]:
offline_users = set(offline["User_id"].dropna().unique())
online_users = set(online["User_id"].dropna().unique())

shared_users = offline_users & online_users

print("Offline users:", len(offline_users))
print("Online users:", len(online_users))
print("Shared users:", len(shared_users))

print(
    "Percent of offline users also online:",
    round(len(shared_users) / len(offline_users) * 100, 2),
    "%"
)

Offline users: 539438
Online users: 762858
Shared users: 267448
Percent of offline users also online: 49.58 %


## 11. Merchant ID Overlap

Verify whether merchant identifiers overlap between the online and offline datasets.

In [17]:
offline_merchants = set(offline["Merchant_id"].dropna().unique())
online_merchants = set(online["Merchant_id"].dropna().unique())

shared_merchants = offline_merchants & online_merchants

print("Offline merchants:", len(offline_merchants))
print("Online merchants:", len(online_merchants))
print("Shared merchant IDs:", len(shared_merchants))

Offline merchants: 8415
Online merchants: 7999
Shared merchant IDs: 0


## 12. Exact Duplicate Rows

Identify exact duplicate-looking records. Do not remove them yet because identical rows may represent multiple real events when transaction IDs and precise timestamps are unavailable.

In [18]:
print("Offline exact duplicates:", offline.duplicated().sum())
print("Online exact duplicates:", online.duplicated().sum())

Offline exact duplicates: 37893
Online exact duplicates: 5607283


## 13. User Activity Density

Inspect how much behavioral history users have. This will help determine which users have enough information to include in the later promotion-abuse model.

In [19]:
offline_user_counts = offline.groupby("User_id").size()
online_user_counts = online.groupby("User_id").size()

print("OFFLINE USER EVENT COUNTS")
display(offline_user_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nONLINE USER EVENT COUNTS")
display(online_user_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

OFFLINE USER EVENT COUNTS


count    539438.000000
mean          3.253171
std           4.767551
min           1.000000
50%           2.000000
75%           3.000000
90%           7.000000
95%          11.000000
99%          23.000000
max         264.000000
dtype: float64


ONLINE USER EVENT COUNTS


count    762858.000000
mean         14.982901
std          34.890628
min           1.000000
50%           7.000000
75%          16.000000
90%          33.000000
95%          53.000000
99%         127.000000
max        5786.000000
dtype: float64